# Real-time / pseudo-real-time macro panel for DNS + GP correction

This notebook builds a **monthly beginning-of-month panel** for the DNS / Macro-DNS / GP project.

It uses a **hybrid data policy**:

- **ALFRED snapshot pulls** for revised macro series where vintage information matters and is available.
- **FRED pulls** for observed financial / rate series and for series that are not exposed through ALFRED vintage queries.
- **First business day of each month** as the model snapshot date.
- For each snapshot date, the panel uses the **latest observation strictly before the snapshot date**.

## Included variables

Original macro block:
- Capacity utilization (`CU`)
- Federal funds rate (`FFR`)
- PCE price index (`PCEPI`)
- 12-month PCE inflation (`INFL`)

Additional GP block:
- Effective fed funds rate (`EFFR`)
- 10Y minus 2Y spread (`spread_10y2y`)
- Moody's Baa yield (`baa_yield`)
- Moody's Aaa yield (`aaa_yield`)
- Major-currency / advanced foreign economies dollar index (`major_dollar`)
- Global 10Y sovereign yield first principal component (`global10y_pc1`)

## Important limitation

ALFRED does **not** provide a usable vintage history for every FRED series. So this notebook uses:
- **ALFRED** for `CUMFNS` and `PCEPI`
- **FRED** for the rest

That is the clean, defensible compromise for this project.


In [1]:

# =========================
# 1. Imports and settings
# =========================
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

# --- user settings ---
API_KEY = "8646e4c441bf17d906a058f8eddaf3f8"
assert API_KEY is not None and API_KEY != "", "Set FRED_API_KEY in your environment before running."

PANEL_START = "1973-01-01"
PANEL_END = pd.Timestamp.today().strftime("%Y-%m-%d")

# Start earlier so inflation can be computed beginning in 1973
MONTHLY_HISTORY_START = "1971-01-01"

OUTDIR = Path("./macro_gp_panel_output")
OUTDIR.mkdir(exist_ok=True)

FRED_OBS_URL = "https://api.stlouisfed.org/fred/series/observations"


In [2]:

# =========================
# 2. Snapshot calendar
# =========================
def first_business_day_snapshots(start=PANEL_START, end=PANEL_END):
    months = pd.date_range(start=start, end=end, freq="MS")
    return pd.DatetimeIndex([m if m.dayofweek < 5 else m + pd.offsets.BDay(1) for m in months], name="snapshot_date")

snapshots = first_business_day_snapshots()
snapshot_df = pd.DataFrame({"snapshot_date": snapshots})
snapshot_df["month"] = snapshot_df["snapshot_date"].dt.to_period("M").astype(str)

print(snapshot_df.head())
print(snapshot_df.tail())


  snapshot_date    month
0    1973-01-01  1973-01
1    1973-02-01  1973-02
2    1973-03-01  1973-03
3    1973-04-02  1973-04
4    1973-05-01  1973-05
    snapshot_date    month
635    2025-12-01  2025-12
636    2026-01-01  2026-01
637    2026-02-02  2026-02
638    2026-03-02  2026-03
639    2026-04-01  2026-04


In [3]:

# =========================
# 3. API helpers
# =========================
def _request_fred(params, max_retries=5, pause=0.5, verbose=False):
    last_response = None
    for attempt in range(max_retries):
        r = requests.get(FRED_OBS_URL, params=params, timeout=60)
        last_response = r
        if r.status_code == 200:
            return r
        if verbose:
            print(f"Attempt {attempt+1} failed: {r.status_code}")
            try:
                print(r.json())
            except Exception:
                print(r.text[:1000])
        time.sleep(pause * (attempt + 1))
    last_response.raise_for_status()


def fetch_fred_monthly_series(series_id, observation_start=MONTHLY_HISTORY_START, observation_end=PANEL_END):
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "observation_start": observation_start,
        "observation_end": observation_end,
        "sort_order": "asc",
    }

    r = _request_fred(params)
    obs = r.json()["observations"]
    df = pd.DataFrame(obs)

    if df.empty:
        return pd.DataFrame(columns=["date", series_id])

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df[series_id] = pd.to_numeric(df["value"], errors="coerce")

    return df[["date", series_id]].sort_values("date").reset_index(drop=True)

def fetch_fred_series(series_id, observation_start, observation_end):
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "observation_start": observation_start,
        "observation_end": observation_end,
        "sort_order": "asc",
    }
    r = _request_fred(params)
    obs = r.json()["observations"]
    df = pd.DataFrame(obs)
    if df.empty:
        return pd.DataFrame(columns=["date", series_id])

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df[series_id] = pd.to_numeric(df["value"], errors="coerce")
    return df[["date", series_id]].sort_values("date").reset_index(drop=True)


def fetch_alfred_snapshot(series_id, snapshot_date, observation_start=MONTHLY_HISTORY_START):
    """
    Pull the observation history for a series as it was known on snapshot_date.
    Works only for series that actually exist in ALFRED.
    """
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "observation_start": observation_start,
        "observation_end": snapshot_date,
        "realtime_start": snapshot_date,
        "realtime_end": snapshot_date,
        "sort_order": "asc",
    }

    r = _request_fred(params)
    obs = r.json()["observations"]
    df = pd.DataFrame(obs)
    if df.empty:
        return pd.DataFrame(columns=["date", series_id])

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df[series_id] = pd.to_numeric(df["value"], errors="coerce")
    return df[["date", series_id]].sort_values("date").reset_index(drop=True)


## Series map

`alfred_monthly_series` are revised monthly macro series we query snapshot-by-snapshot.

`fred_monthly_series` are monthly series we treat as observed / unrevised enough for this project, or not exposed through ALFRED vintage queries.

`fred_daily_series` are daily financial series reduced to the last available value before each monthly snapshot.


In [7]:

# =========================
# 4. Series definitions
# =========================
alfred_monthly_series = {
    "CUMFNS": "Capacity Utilization: Manufacturing",
    "PCEPI": "PCE Price Index",
}

fred_monthly_series = {
    "FEDFUNDS": "Federal Funds Rate",
    "BAA": "Moody's Baa Corporate Bond Yield",
    "AAA": "Moody's Aaa Corporate Bond Yield",
    "DTWEXM": "Major-Currency Dollar Index (discontinued)",
    "TWEXAFEGSMTH": "Advanced Foreign Economies Dollar Index",
    "IRLTLT01CAM156N": "Canada 10Y sovereign yield",
    "IRLTLT01DEM156N": "Germany 10Y sovereign yield",
    "IRLTLT01GBM156N": "UK 10Y sovereign yield",
    "IRLTLT01FRM156N": "France 10Y sovereign yield",
}

fred_daily_series = {
    "DFF": "Effective Federal Funds Rate",
    "T10Y2Y": "10Y minus 2Y Treasury spread",
}


In [5]:

# =========================
# 5. Panel builders
# =========================
ALFRED_START = pd.Timestamp("2006-01-01")

def build_monthly_panel_from_alfred_api(series_id, snapshots, observation_start=MONTHLY_HISTORY_START):
    rows = []

    # Pull ordinary FRED history once as fallback / pre-2006 source
    fred_hist = fetch_fred_monthly_series(
        series_id,
        observation_start=observation_start,
        observation_end=PANEL_END
    )

    for snap in snapshots:
        snap = pd.Timestamp(snap)

        # Before ALFRED era: fall back to ordinary historical series
        if snap < ALFRED_START:
            df = fred_hist.copy()
            df = df[df["date"] < snap].dropna(subset=[series_id])

        else:
            # ALFRED era: use vintage snapshot
            try:
                df = fetch_alfred_snapshot(
                    series_id,
                    snap.strftime("%Y-%m-%d"),
                    observation_start=observation_start
                )
                df = df[df["date"] < snap].dropna(subset=[series_id])
            except requests.HTTPError:
                # graceful fallback if ALFRED still chokes on a given month
                df = fred_hist.copy()
                df = df[df["date"] < snap].dropna(subset=[series_id])

        if df.empty:
            obs_date = pd.NaT
            value = np.nan
        else:
            last_row = df.sort_values("date").iloc[-1]
            obs_date = last_row["date"]
            value = last_row[series_id]

        rows.append({
            "snapshot_date": snap,
            f"{series_id}_obs_date": obs_date,
            series_id: value
        })

    return pd.DataFrame(rows)


def build_monthly_panel_from_fred_history(series_id, hist_df, snapshots):
    rows = []
    for snap in snapshots:
        snap = pd.Timestamp(snap)
        valid = hist_df[hist_df["date"] < snap].dropna(subset=[series_id])

        if valid.empty:
            obs_date = pd.NaT
            value = np.nan
        else:
            chosen = valid.sort_values("date").iloc[-1]
            obs_date = chosen["date"]
            value = chosen[series_id]

        rows.append({
            "snapshot_date": snap,
            f"{series_id}_obs_date": obs_date,
            series_id: value
        })
    return pd.DataFrame(rows)


def build_daily_snapshot_panel(series_id, hist_df, snapshots):
    rows = []
    for snap in snapshots:
        snap = pd.Timestamp(snap)
        valid = hist_df[hist_df["date"] < snap].dropna(subset=[series_id])

        if valid.empty:
            obs_date = pd.NaT
            value = np.nan
        else:
            chosen = valid.sort_values("date").iloc[-1]
            obs_date = chosen["date"]
            value = chosen[series_id]

        rows.append({
            "snapshot_date": snap,
            f"{series_id}_obs_date": obs_date,
            series_id: value
        })
    return pd.DataFrame(rows)


In [8]:

# =========================
# 6. Pull monthly FRED histories
# =========================
fred_monthly_histories = {}
for sid in fred_monthly_series:
    print(f"Downloading monthly FRED history for {sid} ...")
    fred_monthly_histories[sid] = fetch_fred_series(
        sid,
        observation_start=MONTHLY_HISTORY_START,
        observation_end=PANEL_END
    )

for sid, df in fred_monthly_histories.items():
    df.to_csv(OUTDIR / f"fred_monthly_history_{sid}.csv", index=False)

print("Saved monthly FRED histories.")


Saved monthly FRED histories.


In [9]:

# =========================
# 7. Pull daily FRED histories
# =========================
fred_daily_histories = {}
for sid in fred_daily_series:
    print(f"Downloading daily FRED history for {sid} ...")
    fred_daily_histories[sid] = fetch_fred_series(
        sid,
        observation_start=PANEL_START,
        observation_end=PANEL_END
    )

for sid, df in fred_daily_histories.items():
    df.to_csv(OUTDIR / f"fred_daily_history_{sid}.csv", index=False)

print("Saved daily FRED histories.")


Saved daily FRED histories.


In [10]:

# =========================
# 8. Build snapshot panels
# =========================
monthly_panels = {}

# ALFRED-based monthly panels
for sid in alfred_monthly_series:
    print(f"Building ALFRED snapshot panel for {sid} ...")
    monthly_panels[sid] = build_monthly_panel_from_alfred_api(sid, snapshots)

# FRED-based monthly panels
for sid, hist in fred_monthly_histories.items():
    print(f"Building FRED monthly snapshot panel for {sid} ...")
    monthly_panels[sid] = build_monthly_panel_from_fred_history(sid, hist, snapshots)

# daily panels
daily_panels = {}
for sid, hist in fred_daily_histories.items():
    print(f"Building daily snapshot panel for {sid} ...")
    daily_panels[sid] = build_daily_snapshot_panel(sid, hist, snapshots)

macro_rt = snapshot_df.copy()

for sid, panel in monthly_panels.items():
    macro_rt = macro_rt.merge(panel, on="snapshot_date", how="left")

for sid, panel in daily_panels.items():
    macro_rt = macro_rt.merge(panel, on="snapshot_date", how="left")

macro_rt.head()


Building ALFRED snapshot panel for CUMFNS ...
Building ALFRED snapshot panel for PCEPI ...
Building FRED monthly snapshot panel for FEDFUNDS ...
Building FRED monthly snapshot panel for BAA ...
Building FRED monthly snapshot panel for AAA ...
Building FRED monthly snapshot panel for DTWEXM ...
Building FRED monthly snapshot panel for TWEXAFEGSMTH ...
Building FRED monthly snapshot panel for IRLTLT01CAM156N ...
Building FRED monthly snapshot panel for IRLTLT01DEM156N ...
Building FRED monthly snapshot panel for IRLTLT01GBM156N ...
Building FRED monthly snapshot panel for IRLTLT01FRM156N ...
Building daily snapshot panel for DFF ...
Building daily snapshot panel for T10Y2Y ...


,snapshot_date,month,CUMFNS_obs_date,CUMFNS,PCEPI_obs_date,PCEPI,FEDFUNDS_obs_date,FEDFUNDS,BAA_obs_date,BAA,...,IRLTLT01DEM156N_obs_date,IRLTLT01DEM156N,IRLTLT01GBM156N_obs_date,IRLTLT01GBM156N,IRLTLT01FRM156N_obs_date,IRLTLT01FRM156N,DFF_obs_date,DFF,T10Y2Y_obs_date,T10Y2Y
0,1973-01-01,1973-01,1972-12-01,86.5759,1972-12-01,21.630,1972-12-01,5.33,1972-12-01,7.93,...,1972-12-01,8.6,1972-12-01,9.36,1972-12-01,8.28,NaT,NaN,NaT,NaN
1,1973-02-01,1973-02,1973-01-01,86.9787,1973-01-01,21.695,1973-01-01,5.94,1973-01-01,7.90,...,1973-01-01,8.6,1973-01-01,9.17,1973-01-01,8.33,1973-01-31,6.50,NaT,NaN
2,1973-03-01,1973-03,1973-02-01,88.0795,1973-02-01,21.809,1973-02-01,6.58,1973-02-01,7.97,...,1973-02-01,8.6,1973-02-01,9.33,1973-02-01,8.38,1973-02-28,7.50,NaT,NaN
3,1973-04-02,1973-04,1973-04-01,87.5274,1973-04-01,22.127,1973-04-01,7.12,1973-04-01,8.09,...,1973-04-01,8.8,1973-04-01,9.66,1973-04-01,8.62,1973-04-01,7.38,NaT,NaN
4,1973-05-01,1973-05,1973-04-01,87.5274,1973-04-01,22.127,1973-04-01,7.12,1973-04-01,8.09,...,1973-04-01,8.8,1973-04-01,9.66,1973-04-01,8.62,1973-04-30,7.63,NaT,NaN


In [11]:

# =========================
# 9. Inflation construction
# =========================
# Construct beginning-of-month year-over-year inflation from the real-time / pseudo-real-time
# PCEPI panel already aligned to snapshot dates.
macro_rt = macro_rt.sort_values("snapshot_date").reset_index(drop=True)
macro_rt["INFL"] = 100 * (macro_rt["PCEPI"] / macro_rt["PCEPI"].shift(12) - 1)

macro_rt[["snapshot_date", "PCEPI", "INFL"]].head(15)


,snapshot_date,PCEPI,INFL
0,1973-01-01,21.630,NaN
1,1973-02-01,21.695,NaN
2,1973-03-01,21.809,NaN
3,1973-04-02,22.127,NaN
4,1973-05-01,22.127,NaN
5,1973-06-01,22.236,NaN
6,1973-07-02,22.444,NaN
7,1973-08-01,22.444,NaN
8,1973-09-03,22.793,NaN
9,1973-10-01,22.793,NaN


In [12]:

# =========================
# 10. Dollar index splice
# =========================
# Use DTWEXM when available, otherwise splice in TWEXAFEGSMTH using overlap scaling.
overlap = macro_rt[["snapshot_date", "DTWEXM", "TWEXAFEGSMTH"]].dropna()

if overlap.empty:
    macro_rt["major_dollar"] = macro_rt["DTWEXM"].combine_first(macro_rt["TWEXAFEGSMTH"])
    scale = np.nan
else:
    scale = overlap["DTWEXM"].mean() / overlap["TWEXAFEGSMTH"].mean()
    macro_rt["major_dollar"] = np.where(
        macro_rt["DTWEXM"].notna(),
        macro_rt["DTWEXM"],
        macro_rt["TWEXAFEGSMTH"] * scale
    )

print("Dollar splice scale:", scale)
macro_rt[["snapshot_date", "DTWEXM", "TWEXAFEGSMTH", "major_dollar"]].tail()


Dollar splice scale: 0.828278475229712


,snapshot_date,DTWEXM,TWEXAFEGSMTH,major_dollar
635,2025-12-01,90.8221,113.0054,90.8221
636,2026-01-01,90.8221,111.5080,90.8221
637,2026-02-02,90.8221,110.0657,90.8221
638,2026-03-02,90.8221,111.9724,90.8221
639,2026-04-01,90.8221,111.9724,90.8221


In [13]:

# =========================
# 11. Global 10Y sovereign yield PC1
# =========================
global_cols = {
    "IRLTLT01CAM156N": "can10",
    "IRLTLT01DEM156N": "deu10",
    "IRLTLT01GBM156N": "gbr10",
    "IRLTLT01FRM156N": "fra10",
}

for old, new in global_cols.items():
    macro_rt[new] = macro_rt[old]

global_df = macro_rt[["snapshot_date", "can10", "deu10", "gbr10", "fra10"]].dropna().copy()

if len(global_df) > 0:
    scaler = StandardScaler()
    X_std = scaler.fit_transform(global_df[["can10", "deu10", "gbr10", "fra10"]])
    pca = PCA(n_components=1)
    global_df["global10y_pc1"] = pca.fit_transform(X_std)[:, 0]
    explained = pca.explained_variance_ratio_[0]
else:
    global_df["global10y_pc1"] = np.nan
    explained = np.nan

macro_rt = macro_rt.merge(global_df[["snapshot_date", "global10y_pc1"]], on="snapshot_date", how="left")
print("Explained variance ratio of PC1:", explained)
macro_rt[["snapshot_date", "can10", "deu10", "gbr10", "fra10", "global10y_pc1"]].tail()


Explained variance ratio of PC1: 0.9578052495396073


,snapshot_date,can10,deu10,gbr10,fra10,global10y_pc1
635,2025-12-01,3.184286,2.657500,4.4985,3.44,-1.404794
636,2026-01-01,3.422667,2.814211,4.4826,3.56,-1.334909
637,2026-02-02,3.288421,2.745000,4.4324,3.40,-1.388857
638,2026-03-02,3.440000,2.905238,4.7007,3.40,-1.310375
639,2026-04-01,3.440000,2.905238,4.7007,3.40,-1.310375


In [14]:

# =========================
# 12. Final panel
# =========================
final = macro_rt.rename(columns={
    "FEDFUNDS": "FFR",
    "CUMFNS": "CU",
    "DFF": "EFFR",
    "T10Y2Y": "spread_10y2y",
    "BAA": "baa_yield",
    "AAA": "aaa_yield",
})

keep_cols = [
    "snapshot_date",
    "month",

    # Original three macro indicators
    "CU",
    "FFR",
    "PCEPI",
    "INFL",

    # Additional GP block
    "EFFR",
    "spread_10y2y",
    "baa_yield",
    "aaa_yield",
    "major_dollar",
    "global10y_pc1",

    # Observation metadata
    "CUMFNS_obs_date",
    "FEDFUNDS_obs_date",
    "PCEPI_obs_date",
    "DFF_obs_date",
    "T10Y2Y_obs_date",
    "BAA_obs_date",
    "AAA_obs_date",
    "DTWEXM_obs_date",
    "TWEXAFEGSMTH_obs_date",
    "IRLTLT01CAM156N_obs_date",
    "IRLTLT01DEM156N_obs_date",
    "IRLTLT01GBM156N_obs_date",
    "IRLTLT01FRM156N_obs_date",
]

final = final[keep_cols].copy()
final.head(20)


,snapshot_date,month,CU,FFR,PCEPI,INFL,EFFR,spread_10y2y,baa_yield,aaa_yield,...,DFF_obs_date,T10Y2Y_obs_date,BAA_obs_date,AAA_obs_date,DTWEXM_obs_date,TWEXAFEGSMTH_obs_date,IRLTLT01CAM156N_obs_date,IRLTLT01DEM156N_obs_date,IRLTLT01GBM156N_obs_date,IRLTLT01FRM156N_obs_date
0,1973-01-01,1973-01,86.5759,5.33,21.630,NaN,NaN,NaN,7.93,7.08,...,NaT,NaT,1972-12-01,1972-12-01,NaT,NaT,1972-12-01,1972-12-01,1972-12-01,1972-12-01
1,1973-02-01,1973-02,86.9787,5.94,21.695,NaN,6.50,NaN,7.90,7.15,...,1973-01-31,NaT,1973-01-01,1973-01-01,1973-01-31,NaT,1973-01-01,1973-01-01,1973-01-01,1973-01-01
2,1973-03-01,1973-03,88.0795,6.58,21.809,NaN,7.50,NaN,7.97,7.22,...,1973-02-28,NaT,1973-02-01,1973-02-01,1973-02-28,NaT,1973-02-01,1973-02-01,1973-02-01,1973-02-01
3,1973-04-02,1973-04,87.5274,7.12,22.127,NaN,7.38,NaN,8.09,7.26,...,1973-04-01,NaT,1973-04-01,1973-04-01,1973-03-30,NaT,1973-04-01,1973-04-01,1973-04-01,1973-04-01
4,1973-05-01,1973-05,87.5274,7.12,22.127,NaN,7.63,NaN,8.09,7.26,...,1973-04-30,NaT,1973-04-01,1973-04-01,1973-04-30,NaT,1973-04-01,1973-04-01,1973-04-01,1973-04-01
5,1973-06-01,1973-06,87.8140,7.84,22.236,NaN,8.13,NaN,8.06,7.29,...,1973-05-31,NaT,1973-05-01,1973-05-01,1973-05-31,NaT,1973-05-01,1973-05-01,1973-05-01,1973-05-01
6,1973-07-02,1973-07,87.5919,10.40,22.444,NaN,8.88,NaN,8.24,7.45,...,1973-07-01,NaT,1973-07-01,1973-07-01,1973-06-28,NaT,1973-07-01,1973-07-01,1973-07-01,1973-07-01
7,1973-08-01,1973-08,87.5919,10.40,22.444,NaN,11.22,NaN,8.24,7.45,...,1973-07-31,NaT,1973-07-01,1973-07-01,1973-07-31,NaT,1973-07-01,1973-07-01,1973-07-01,1973-07-01
8,1973-09-03,1973-09,87.4473,10.78,22.793,NaN,11.22,NaN,8.63,7.63,...,1973-09-02,NaT,1973-09-01,1973-09-01,1973-08-31,NaT,1973-09-01,1973-09-01,1973-09-01,1973-09-01
9,1973-10-01,1973-10,87.4473,10.78,22.793,NaN,10.82,NaN,8.63,7.63,...,1973-09-30,NaT,1973-09-01,1973-09-01,1973-09-28,NaT,1973-09-01,1973-09-01,1973-09-01,1973-09-01


In [15]:

# =========================
# 13. Diagnostics
# =========================
display_cols = [
    "snapshot_date", "CU", "FFR", "PCEPI", "INFL",
    "EFFR", "spread_10y2y", "baa_yield", "aaa_yield",
    "major_dollar", "global10y_pc1"
]
print(final[display_cols].head(20))

print("\nMissing values by column:")
print(final.isna().sum().sort_values(ascending=False).head(25))


   snapshot_date       CU    FFR   PCEPI       INFL   EFFR  spread_10y2y  \
0     1973-01-01  86.5759   5.33  21.630        NaN    NaN           NaN   
1     1973-02-01  86.9787   5.94  21.695        NaN   6.50           NaN   
2     1973-03-01  88.0795   6.58  21.809        NaN   7.50           NaN   
3     1973-04-02  87.5274   7.12  22.127        NaN   7.38           NaN   
4     1973-05-01  87.5274   7.12  22.127        NaN   7.63           NaN   
5     1973-06-01  87.8140   7.84  22.236        NaN   8.13           NaN   
6     1973-07-02  87.5919  10.40  22.444        NaN   8.88           NaN   
7     1973-08-01  87.5919  10.40  22.444        NaN  11.22           NaN   
8     1973-09-03  87.4473  10.78  22.793        NaN  11.22           NaN   
9     1973-10-01  87.4473  10.78  22.793        NaN  10.82           NaN   
10    1973-11-01  88.0325  10.01  22.922        NaN  10.61           NaN   
11    1973-12-03  88.0841   9.95  23.300        NaN  10.34           NaN   
12    1974-0

In [16]:

# =========================
# 14. Save outputs
# =========================
final.to_csv(OUTDIR / "macro_gp_panel_realtime_bom_hybrid.csv", index=False)
print(f"Saved: {OUTDIR / 'macro_gp_panel_realtime_bom_hybrid.csv'}")


Saved: macro_gp_panel_output/macro_gp_panel_realtime_bom_hybrid.csv
